# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice

I will use Logistic Regression as the first ML model because this is a binary prediction and ranking problem. The target is `is_declining_label`, which represents an observed yes/no outcome.

Logistic Regression is appropriate because it provides a simple and interpretable probability score for each webpage. These probabilities can be used to rank webpages by their likelihood of declining and prioritize pages for review.

The model will use historical search and engagement features available at the decision point. I will not use `trend_pct`, `trend_direction`, or `is_declining_label` as input features because they are directly related to the target and could cause leakage. `content_id` and `client_id` will also not be used as model features.

The model will be compared against the Week-4 rule-based baseline using the same test data, same split, and precision@K metrics.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split Design

I will use a grouped train/test split based on `client_id`. The same client will not appear in both the training and test sets.

This is an honest split for the research question because webpages belonging to the same client can share similar search, content, and engagement patterns. Allowing the same client in both sets could make the model appear better than it really is.

I will assign 80% of the clients to the training set and 20% to the test set, using a fixed random seed for reproducibility. `client_id` will only be used to create the split and will not be given to the model as a feature.

The final model and the Week-4 baseline will be evaluated on this same held-out test set using the same precision@20 and precision@50 metrics.

In [30]:
# ---------------------------------------------------------
# Section 2: Grouped Train/Test Split
# ---------------------------------------------------------

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# Load dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print("Shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())

# ---------------------------------------------------------
# Grouped split by client
# ---------------------------------------------------------

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# ---------------------------------------------------------
# Check split
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("GROUPED TRAIN / TEST SPLIT")
print("=" * 60)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())

# ---------------------------------------------------------
# Verify no client appears in both sets
# ---------------------------------------------------------

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

overlap = train_clients.intersection(test_clients)

print("Client overlap:", overlap)

DATASET LOADED
Shape: (30000, 44)
Unique clients: 32

GROUPED TRAIN / TEST SPLIT
Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [31]:
# ---------------------------------------------------------
# Section 3: Train + Compare vs My Baseline
# ---------------------------------------------------------

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ---------------------------------------------------------
# Create the target label
# ---------------------------------------------------------
# "down" means the content item is declining.
# 1 = declining
# 0 = not declining

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

train_df["is_declining_label"] = (
    train_df["trend_direction"] == "down"
).astype(int)

test_df["is_declining_label"] = (
    test_df["trend_direction"] == "down"
).astype(int)


# ---------------------------------------------------------
# Verify target
# ---------------------------------------------------------

print("=" * 60)
print("TARGET DISTRIBUTION")
print("=" * 60)

print(df["is_declining_label"].value_counts())

print("\nTrend direction vs target:")
display(
    pd.crosstab(
        df["trend_direction"],
        df["is_declining_label"]
    )
)


# ---------------------------------------------------------
# Select model features
# ---------------------------------------------------------
# These are historical numeric features available before
# making the prediction.
#
# Excluded:
# content_id       -> identifier
# client_id        -> used only for grouped splitting
# trend_direction  -> used to create the target
# trend_pct        -> directly related to the target
# is_declining_label -> target itself

feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

target_column = "is_declining_label"

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df[target_column].copy()
y_test = test_df[target_column].copy()


# ---------------------------------------------------------
# Check missing values
# ---------------------------------------------------------
# Missing numeric values will be handled using the median
# during model preprocessing.

missing_values = X_train.isnull().sum()
missing_values = missing_values[missing_values > 0]

print("\n" + "=" * 60)
print("MISSING VALUES IN TRAINING FEATURES")
print("=" * 60)

display(
    missing_values.sort_values(ascending=False)
)


# ---------------------------------------------------------
# Build Logistic Regression model
# ---------------------------------------------------------
# Median imputation handles missing values.
# StandardScaler puts features on comparable scales.
# Logistic Regression provides an interpretable probability
# that a page belongs to the declining class.

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])


# ---------------------------------------------------------
# Train the model
# ---------------------------------------------------------

logistic_model.fit(X_train, y_train)

print("\n" + "=" * 60)
print("LOGISTIC REGRESSION TRAINED")
print("=" * 60)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Number of features:", len(feature_columns))


# ---------------------------------------------------------
# Generate model ranking scores
# ---------------------------------------------------------
# The probability of class 1 (declining) is used as the
# ranking score.

model_scores = logistic_model.predict_proba(X_test)[:, 1]

model_predictions = (
    model_scores >= 0.5
).astype(int)

print("\n" + "=" * 60)
print("MODEL PREDICTIONS")
print("=" * 60)

print("Number of predictions:", len(model_scores))
print("Minimum score:", round(model_scores.min(), 4))
print("Maximum score:", round(model_scores.max(), 4))
print("Average score:", round(model_scores.mean(), 4))


# ---------------------------------------------------------
# Precision@K function
# ---------------------------------------------------------
# Precision@K measures how many of the top K ranked pages
# are actually declining.

def precision_at_k(scores, labels, k):

    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)

    return labels[order[:k]].mean()


# ---------------------------------------------------------
# Logistic Regression Precision@K
# ---------------------------------------------------------

model_p20 = precision_at_k(
    model_scores,
    y_test,
    20
)

model_p50 = precision_at_k(
    model_scores,
    y_test,
    50
)

base_rate = y_test.mean()


# ---------------------------------------------------------
# Recreate ML-07 baseline on the SAME test set
# ---------------------------------------------------------
# Thresholds are calculated using training data only.
# This prevents the test set from influencing the baseline.

baseline_test = test_df.copy()

impression_threshold = (
    train_df["impressions_90d"].mean()
)

ctr_threshold = (
    train_df["ctr"].mean()
)

position_threshold = 20


# ---------------------------------------------------------
# ML-07 baseline score
# ---------------------------------------------------------
# +1 = high impressions
# +1 = low CTR
# +1 = poor average position

baseline_test["baseline_score"] = (
    (baseline_test["impressions_90d"] > impression_threshold).astype(int)
    + (baseline_test["ctr"] < ctr_threshold).astype(int)
    + (baseline_test["avg_position"] > position_threshold).astype(int)
)


# ---------------------------------------------------------
# Baseline Precision@K
# ---------------------------------------------------------

baseline_scores = (
    baseline_test["baseline_score"].values
)

baseline_p20 = precision_at_k(
    baseline_scores,
    y_test,
    20
)

baseline_p50 = precision_at_k(
    baseline_scores,
    y_test,
    50
)


# ---------------------------------------------------------
# Final comparison table
# ---------------------------------------------------------
# Both methods are evaluated on:
# - the same test set
# - the same target
# - the same metrics

comparison = pd.DataFrame({
    "Method": [
        "ML-07 Rule Baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        baseline_p20,
        model_p20
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ],
    "Base Rate": [
        base_rate,
        base_rate
    ]
})


# ---------------------------------------------------------
# Display final results
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("MODEL vs ML-07 BASELINE")
print("=" * 60)

display(
    comparison.round(4)
)

TARGET DISTRIBUTION
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Trend direction vs target:


is_declining_label,0,1
trend_direction,,
down,0,16262
flat,1152,0
new,2236,0
stable,5962,0
up,4388,0



MISSING VALUES IN TRAINING FEATURES


char_count       6614
word_count       6614
competition      2319
search_volume    2319
cpc              2319
scroll_rate        15
dtype: int64


LOGISTIC REGRESSION TRAINED
Training rows: 23837
Testing rows: 6163
Number of features: 28

MODEL PREDICTIONS
Number of predictions: 6163
Minimum score: 0.0
Maximum score: 1.0
Average score: 0.4857

MODEL vs ML-07 BASELINE


,Method,Precision@20,Precision@50,Base Rate
0,ML-07 Rule Baseline,0.55,0.48,0.511
1,Logistic Regression,1.00,1.00,0.511


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [34]:
# ---------------------------------------------------------
# Section 4: Errors and Interpretation
# ---------------------------------------------------------

# ---------------------------------------------------------
# Find the most important features
# ---------------------------------------------------------
# Logistic Regression uses coefficients to show how strongly
# each feature is associated with the probability of decline.
#
# Positive coefficient -> higher value increases the predicted
# probability of decline.
#
# Negative coefficient -> higher value decreases the predicted
# probability of decline.

coefficients = (
    logistic_model
    .named_steps["model"]
    .coef_[0]
)

feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": coefficients
})

feature_importance["absolute_coefficient"] = (
    feature_importance["coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    "absolute_coefficient",
    ascending=False
).reset_index(drop=True)


# ---------------------------------------------------------
# Display the top features
# ---------------------------------------------------------

print("=" * 60)
print("TOP 10 FEATURES USED BY THE MODEL")
print("=" * 60)

display(
    feature_importance[
        ["feature", "coefficient"]
    ].head(10)
)

print("=" * 60)
print("TOP 3 FEATURES")
print("=" * 60)

display(
    feature_importance[
        ["feature", "coefficient"]
    ].head(3)
)


# ---------------------------------------------------------
# Create a table of test predictions
# ---------------------------------------------------------
# This lets us inspect where the model made incorrect
# predictions.

error_analysis = test_df.copy()

error_analysis["predicted_probability"] = model_scores

error_analysis["predicted_label"] = (
    model_predictions
)

error_analysis["actual_label"] = (
    y_test.values
)

error_analysis["correct"] = (
    error_analysis["predicted_label"]
    == error_analysis["actual_label"]
)


# ---------------------------------------------------------
# Find incorrect predictions
# ---------------------------------------------------------

wrong_predictions = error_analysis[
    error_analysis["correct"] == False
].copy()


print("\n" + "=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

print(
    "Total test cases:",
    len(error_analysis)
)

print(
    "Incorrect predictions:",
    len(wrong_predictions)
)

print(
    "Error rate:",
    round(
        len(wrong_predictions) / len(error_analysis),
        4
    )
)


# ---------------------------------------------------------
# Show three concrete wrong cases
# ---------------------------------------------------------
# These examples help us understand why some pages are
# difficult for the model to classify.

wrong_predictions["prediction_confidence"] = (
    np.abs(
        wrong_predictions["predicted_probability"] - 0.5
    )
)

three_wrong_cases = (
    wrong_predictions
    .sort_values(
        "prediction_confidence"
    )
    .head(3)
)


print("\n" + "=" * 60)
print("THREE CONCRETE WRONG CASES")
print("=" * 60)

display(
    three_wrong_cases[
        [
            "content_id",
            "client_id",
            "trend_direction",
            "trend_pct",
            "predicted_probability",
            "predicted_label",
            "actual_label",
            "avg_position",
            "ctr",
            "impressions_90d"
        ]
    ]
)


# ---------------------------------------------------------
# Error summary by actual class
# ---------------------------------------------------------
# This shows whether the model struggles more with declining
# pages or non-declining pages.

print("\n" + "=" * 60)
print("ERROR SUMMARY BY ACTUAL CLASS")
print("=" * 60)

error_summary = (
    error_analysis
    .groupby("actual_label")
    .agg(
        total=("actual_label", "size"),
        errors=("correct", lambda x: (~x).sum())
    )
)

error_summary["error_rate"] = (
    error_summary["errors"]
    / error_summary["total"]
)

display(
    error_summary.round(4)
)


# ---------------------------------------------------------
# Interpretation
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)

print(
    "The top three features show which historical signals "
    "the Logistic Regression model relies on most strongly."
)

print(
    "The three wrong cases demonstrate that some webpages "
    "are difficult to classify using historical aggregate "
    "signals alone."
)

print(
    "Possible sources of error include unusual page behaviour, "
    "missing information, recent changes, or patterns that "
    "are not captured by the selected features."
)

print(
    "The error analysis should be reviewed before concluding "
    "that the model is better than the rule-based baseline."
)

# ---------------------------------------------------------
# Final Leakage Check
# ---------------------------------------------------------

print("=" * 60)
print("FINAL FEATURE LEAKAGE CHECK")
print("=" * 60)

# These fields must never be used as model features
forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]

# Check whether any forbidden field is present
# in the features used by the model
used_forbidden = [
    feature
    for feature in forbidden_features
    if feature in feature_columns
]

print("Features used by the model:")
print(feature_columns)

print("\nForbidden features found in model:")
print(used_forbidden)

# Final result
if len(used_forbidden) == 0:
    print("\nPASS: No label-derived fields or IDs are used as model features.")
else:
    print("\nWARNING: Remove these fields before final submission.")

TOP 10 FEATURES USED BY THE MODEL


,feature,coefficient
0,impressions_last_30d,-35.303909
1,impressions_prev_30d,29.284556
2,impressions_90d,1.509282
3,clicks_last_30d,-0.879000
4,clicks_prev_30d,0.809266
5,sessions_last_30d,-0.792680
6,sessions_90d,0.729695
7,pageviews_90d,0.718496
8,users_90d,-0.693257
9,days_with_impressions,0.545067


TOP 3 FEATURES


,feature,coefficient
0,impressions_last_30d,-35.303909
1,impressions_prev_30d,29.284556
2,impressions_90d,1.509282



ERROR ANALYSIS
Total test cases: 6163
Incorrect predictions: 1503
Error rate: 0.2439

THREE CONCRETE WRONG CASES


,content_id,client_id,trend_direction,trend_pct,predicted_probability,predicted_label,actual_label,avg_position,ctr,impressions_90d
21584,content_a16d664f3f07,client_e629fa6598,stable,-13.3,0.500102,1,0,17.7,0.25,1196
19575,content_c8e7e069617c,client_f369cb89fc,up,500.0,0.500103,1,0,8.6,0.00,7
10278,content_2a04fe3892bc,client_4e07408562,stable,-15.5,0.500133,1,0,15.3,0.17,4101



ERROR SUMMARY BY ACTUAL CLASS


,total,errors,error_rate
actual_label,,,
0,3014,518,0.1719
1,3149,985,0.3128



INTERPRETATION
The top three features show which historical signals the Logistic Regression model relies on most strongly.
The three wrong cases demonstrate that some webpages are difficult to classify using historical aggregate signals alone.
Possible sources of error include unusual page behaviour, missing information, recent changes, or patterns that are not captured by the selected features.
The error analysis should be reviewed before concluding that the model is better than the rule-based baseline.
FINAL FEATURE LEAKAGE CHECK
Features used by the model:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', '

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.